# MarketShield · IndoBERT hoax filter

Fine-tunes **IndoBERT** (`indobenchmark/indobert-base-p1`) to separate Indonesian hoax messages from factual ones, with an honest evaluation against a TF-IDF baseline.

Runtime → Change runtime type → **T4 GPU**. Total time ≈ 10 minutes.

**Security:** never paste keys into a cell. The optional Firestore step reads the service-account JSON from **Colab Secrets** (key icon, name `MS_FIRESTORE_KEY`). Share this notebook only with named people.

In [ ]:
!pip -q install transformers datasets scikit-learn accelerate
!git clone -q https://github.com/aarieffawwaz/marketshield.git
%cd marketshield

## 1. Train and evaluate
Duplicates are removed before the 80 / 10 / 10 split, the best epoch is picked on validation, and the test split is used once. Results go to `results/`.

In [ ]:
!python ml/train_hoax.py --epochs 2 --out results --save-model model

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
r = json.load(open('results/hoax_metrics.json'))
display(pd.DataFrame({
    'TF-IDF + LogReg': r['baseline_tfidf_logreg']['test'],
    'IndoBERT': r['indobert']['test'],
    'IndoBERT · disaster subset': r['indobert']['test_disaster'],
}).T)
display(Image('results/hoax_results.png'))

## 2. Try a message

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tok = AutoTokenizer.from_pretrained('model'); model = AutoModelForSequenceClassification.from_pretrained('model').eval()

def check(text):
    # the training data is lower-cased without punctuation, so normalise the same way
    import re
    t = re.sub(r'[^0-9a-z ]', ' ', re.sub(r'http\S+|@\w+|#\w+', '', text.lower()))
    with torch.no_grad():
        p = torch.softmax(model(**tok(t, return_tensors='pt', truncation=True, max_length=128)).logits, -1)[0]
    return {'label': model.config.id2label[int(p.argmax())], 'p_hoax': round(float(p[1]), 3)}

check('Katanya lahar sudah masuk Pronojiwo, segera tinggalkan rumah! Sebarkan ke grup lain!!!')

## 3. (Optional) Write an outcome to Firestore
Follows `firebase/ML-CONTRACT.md`: the `signals` document stores the outcome only, **never the message text**.

In [ ]:
import json, firebase_admin
from firebase_admin import credentials, firestore
from google.colab import userdata

if not firebase_admin._apps:
    firebase_admin.initialize_app(credentials.Certificate(json.loads(userdata.get('MS_FIRESTORE_KEY'))))
db = firestore.client()

msg = 'Katanya lahar sudah masuk Pronojiwo, segera tinggalkan rumah! Sebarkan ke grup lain!!!'
res = check(msg)
db.collection('signals').add({
    'label': 'HOAX' if res['label'] == 'HOAKS' else 'VERIFIED',
    'confidence': res['p_hoax'] if res['label'] == 'HOAKS' else 1 - res['p_hoax'],
    'triggers': [], 'lang': 'id', 'length': len(msg),
    'model': 'indobert-hoax-v1', 'source': 'colab',
    'createdAt': firestore.SERVER_TIMESTAMP,
})
print('written', res)